# Macenko stain normalization demo

Stain normalization is the technique that distinguishes a serious histology project from a beginner one. This notebook:

1. Picks a clean reference H&E image from the training split.
2. Fits a `MacenkoNormalizer` to it.
3. Shows source -> normalized side-by-side for images from a few different patients (so you can see staining drift get corrected).
4. Splits each image into its hematoxylin and eosin pseudo-channels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from src.dataset import patient_level_split, scan_directory, to_dataframe
from src.stain_norm import MacenkoNormalizer

DATA_ROOT = '../data/BreaKHis_v1'
df = to_dataframe(scan_directory(DATA_ROOT))
splits = patient_level_split(df, val_size=0.15, test_size=0.15, random_state=42)
train = splits['train']
print(f'train: {len(train):,} images / {train.patient_id.nunique()} patients')

In [ ]:
# Pick a 'clean' reference -- a 200X malignant ductal carcinoma image is a common choice.
ref_pool = train[(train.subtype == 'DC') & (train.magnification == 200)]
ref_row = ref_pool.sample(1, random_state=0).iloc[0]
ref_rgb = np.array(Image.open(ref_row.path).convert('RGB'))

norm = MacenkoNormalizer().fit(ref_rgb)
print(f'fitted reference: {ref_row.filename}')
print(f'target stain matrix:\n{norm.target_stain.round(3)}')
print(f'target max concentrations: {norm.target_max_c.round(3)}')

plt.figure(figsize=(4, 4)); plt.imshow(ref_rgb); plt.title('Reference'); plt.axis('off');

In [ ]:
# Show 6 source images from different patients alongside their normalized version.
samples = train.groupby('patient_id').sample(1, random_state=1).sample(6, random_state=2)
fig, axes = plt.subplots(6, 2, figsize=(8, 18))
for i, (_, row) in enumerate(samples.iterrows()):
    src = np.array(Image.open(row.path).convert('RGB'))
    out = norm.transform(src)
    axes[i, 0].imshow(src); axes[i, 0].set_title(f'src ({row.patient_id})'); axes[i, 0].axis('off')
    axes[i, 1].imshow(out); axes[i, 1].set_title('macenko-normalized'); axes[i, 1].axis('off')
plt.tight_layout(); plt.show()

In [ ]:
# Hematoxylin / eosin channel split
demo = train.sample(1, random_state=3).iloc[0]
img = np.array(Image.open(demo.path).convert('RGB'))
H, E = norm.split_stains(img)
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
axes[0].imshow(img); axes[0].set_title('original');                axes[0].axis('off')
axes[1].imshow(H);   axes[1].set_title('Hematoxylin (nuclei)');    axes[1].axis('off')
axes[2].imshow(E);   axes[2].set_title('Eosin (cytoplasm/stroma)'); axes[2].axis('off')
plt.tight_layout(); plt.show()